In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_sample_submission.csv
/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_detailed_class_info.csv
/kaggle/input/competitions/rsna-pneumonia-detection-challenge/GCP Credits Request Link - RSNA.txt
/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv


In [ ]:

import os
from pathlib import Path

for root, dirs, files in os.walk("/kaggle/input"):
    if "stage_2_train_labels.csv" in files:
        print("Labels:", os.path.join(root, "stage_2_train_labels.csv"))
    if "stage_2_train_images" in dirs:
        print("Images:", os.path.join(root, "stage_2_train_images"))


In [ ]:

!pip install -q pydicom opencv-python-headless scikit-learn


In [ ]:

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import pydicom
import cv2
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report
)

SEED = 42
SIZE = 224
BATCH_SIZE = 16

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
!nvidia-smi

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))
print("Visible CPUs:", tf.config.list_physical_devices("CPU"))

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

with tf.device("/GPU:0"):
    x = tf.random.normal((2000, 2000))
    y = tf.matmul(x, x)

print("GPU computation successful:", y.shape)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 3:
        print(f"\nFolder: {root}")

        for file in files[:10]:
            print("  ", file)

        if level >= 3:
            dirs[:] = []

In [ ]:
import tensorflow as tf

print("GPUs:", tf.config.list_physical_devices("GPU"))
print("TensorFlow is ready.")

In [ ]:
import os

base = "/kaggle/input"

for name in os.listdir(base):
    path = os.path.join(base, name)
    print(f"\nDataset: {name}")

    if os.path.isfile(path):
        print("File:", path)
    else:
        for item in os.listdir(path):
            print("  ", item)

In [ ]:
import os

base = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge"

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)

    print(f"\nFolder: {root}")
    print("Files:", len(files))

    for file in files[:8]:
        print("  ", file)

    if level >= 2:
        dirs[:] = []

In [ ]:
import os

base = "/kaggle/input"

print("Datasets attached:", os.listdir(base))

for root, dirs, files in os.walk(base):
    for file in files:
        if file.endswith((".csv", ".zip")):
            print(os.path.join(root, file))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "stage_2_train_labels.csv" in files:
        print("Labels CSV:", os.path.join(root, "stage_2_train_labels.csv"))

    if "stage_2_train_images" in dirs:
        print("DICOM images:", os.path.join(root, "stage_2_train_images"))

In [ ]:
import os

print("Input directory exists:", os.path.exists("/kaggle/input"))
print("\nDatasets attached to this notebook:")

if os.path.exists("/kaggle/input"):
    for item in os.listdir("/kaggle/input"):
        print("-", item)
else:
    print("No /kaggle/input directory found.")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "train_images" in file.lower():
            print(os.path.join(root, file))

In [ ]:
import os

print("Datasets in Kaggle Input:")

for name in os.listdir("/kaggle/input"):
    path = os.path.join("/kaggle/input", name)
    print(f"\n{name}")

    for root, dirs, files in os.walk(path):
        print("Path:", root)
        print("Files:", len(files))

        if root.count(os.sep) - path.count(os.sep) >= 2:
            dirs[:] = []

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 3:
        print(f"\nFolder: {root}")

        for file in files[:10]:
            print("  ", file)

        if level >= 3:
            dirs[:] = []

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "train_images" in file.lower():
            print(os.path.join(root, file))

In [ ]:
import os

DATA_DIR = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge"

LABELS_CSV = os.path.join(DATA_DIR, "stage_2_train_labels.csv")
TRAIN_DIR = os.path.join(DATA_DIR, "stage_2_train_images")
TEST_DIR = os.path.join(DATA_DIR, "stage_2_test_images")

print("Labels CSV exists:", os.path.isfile(LABELS_CSV))
print("Training folder exists:", os.path.isdir(TRAIN_DIR))
print("Test folder exists:", os.path.isdir(TEST_DIR))

train_files = os.listdir(TRAIN_DIR)
print("Training image files:", len(train_files))
print("Example files:", train_files[:5])


In [ ]:
import pandas as pd

labels = pd.read_csv(LABELS_CSV)

print("Dataset shape:", labels.shape)
print("\nColumns:", labels.columns.tolist())
print("\nTarget distribution:")
print(labels.groupby("patientId")["Target"].max().value_counts())

labels.head()

In [ ]:
import os
import pandas as pd

DATA_DIR = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge"

LABELS_CSV = os.path.join(DATA_DIR, "stage_2_train_labels.csv")
TRAIN_DIR = os.path.join(DATA_DIR, "stage_2_train_images")

labels = pd.read_csv(LABELS_CSV)

print("Labels loaded:", len(labels))
print("Columns:", labels.columns.tolist())
print("\nFirst 5 rows:")
display(labels.head())

print("\nUnique patients:", labels["patientId"].nunique())
print("\nPneumonia target counts:")
print(labels.groupby("patientId")["Target"].max().value_counts())

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge"
TRAIN_DIR = os.path.join(DATA_DIR, "stage_2_train_images")
LABELS_CSV = os.path.join(DATA_DIR, "stage_2_train_labels.csv")

# Load labels
labels = pd.read_csv(LABELS_CSV)

# One row per patient; Target=1 if any row has a positive finding
patient_labels = (
    labels.groupby("patientId", as_index=False)["Target"]
    .max()
)

# Verify image availability
patient_labels["image_path"] = patient_labels["patientId"].map(
    lambda pid: os.path.join(TRAIN_DIR, f"{pid}.dcm")
)

patient_labels["image_exists"] = patient_labels["image_path"].map(
    os.path.isfile
)

print("Total patients:", len(patient_labels))
print("Images found:", patient_labels["image_exists"].sum())
print("Images missing:", (~patient_labels["image_exists"]).sum())

# Keep patients whose DICOM images exist
patient_labels = patient_labels[patient_labels["image_exists"]].copy()

# Stratified split: 80% train, 20% validation
train_df, val_df = train_test_split(
    patient_labels,
    test_size=0.20,
    random_state=42,
    stratify=patient_labels["Target"]
)

print("\nTraining patients:", len(train_df))
print("Validation patients:", len(val_df))

print("\nTraining class distribution:")
print(train_df["Target"].value_counts())

print("\nValidation class distribution:")
print(val_df["Target"].value_counts())

In [ ]:
!pip install -q pydicom opencv-python-headless scikit-learn

In [ ]:
import os
import cv2
import pydicom
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Libraries imported successfully.")

In [ ]:
IMG_SIZE = 224

def load_dicom_image(path):
    """Load and preprocess one chest X-ray DICOM image."""
    try:
        dicom = pydicom.dcmread(path)
        image = dicom.pixel_array.astype(np.float32)

        # Apply DICOM rescale settings when present
        slope = float(getattr(dicom, "RescaleSlope", 1))
        intercept = float(getattr(dicom, "RescaleIntercept", 0))
        image = image * slope + intercept

        # Apply MONOCHROME1 inversion when required
        if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
            image = image.max() + image.min() - image

        # Robust intensity scaling
        low, high = np.percentile(image, [1, 99])

        if high > low:
            image = np.clip(image, low, high)
            image = (image - low) / (high - low)
        else:
            image = np.zeros_like(image)

        image = cv2.resize(
            image, (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_AREA
        )

        image = np.repeat(image[..., np.newaxis], 3, axis=-1)
        return image.astype(np.float32)

    except Exception as e:
        print(f"Failed to read {path}: {e}")
        return None

In [ ]:
sample_df = train_df.sample(
    n=min(4, len(train_df)),
    random_state=42
)

fig, axes = plt.subplots(1, len(sample_df), figsize=(12, 4))

if len(sample_df) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    image = load_dicom_image(row["image_path"])

    if image is not None:
        ax.imshow(image[:, :, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(
            "Pneumonia" if row["Target"] == 1 else "No finding"
        )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import random
import cv2
import pydicom
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

from tensorflow.keras.layers import (
    Input,
    Rescaling,
    GlobalAveragePooling2D,
    Dropout,
    Dense
)
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import (
    BinaryAccuracy,
    AUC,
    Precision,
    Recall
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))
print("GPU count:", len(tf.config.list_physical_devices("GPU")))

In [ ]:
import tensorflow as tf
from tensorflow.keras.application import DenseNet121

inputs = layers.Input(shape=(224, 224, 3))

x = layers.Rescaling(
    scale=2.0,
    offset=-1.0
)(inputs)

base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()

In [ ]:
DATA_DIR = (
    "/kaggle/input/competitions/"
    "rsna-pneumonia-detection-challenge"
)

LABELS_CSV = os.path.join(
    DATA_DIR, "stage_2_train_labels.csv"
)

TRAIN_DIR = os.path.join(
    DATA_DIR, "stage_2_train_images"
)

TEST_DIR = os.path.join(
    DATA_DIR, "stage_2_test_images"
)

print("Labels CSV exists:", os.path.isfile(LABELS_CSV))
print("Training directory exists:", os.path.isdir(TRAIN_DIR))
print("Test directory exists:", os.path.isdir(TEST_DIR))

if not os.path.isfile(LABELS_CSV):
    raise FileNotFoundError(f"Labels not found: {LABELS_CSV}")

if not os.path.isdir(TRAIN_DIR):
    raise FileNotFoundError(f"Training images not found: {TRAIN_DIR}")

print("\nExample training files:")
print(os.listdir(TRAIN_DIR)[:5])

In [ ]:
labels = pd.read_csv(LABELS_CSV)

print("Original CSV shape:", labels.shape)
print("Columns:", labels.columns.tolist())

patient_labels = (
    labels.groupby("patientId", as_index=False)["Target"]
    .max()
)

patient_labels["Target"] = patient_labels["Target"].astype("int32")

patient_labels["image_path"] = patient_labels["patientId"].map(
    lambda patient_id: os.path.join(
        TRAIN_DIR, f"{patient_id}.dcm"
    )
)

patient_labels["image_exists"] = (
    patient_labels["image_path"].map(os.path.isfile)
)

print("\nUnique patients:", len(patient_labels))
print("Images found:", int(patient_labels["image_exists"].sum()))
print("Images missing:", int((~patient_labels["image_exists"]).sum()))

print("\nPatient-level class distribution:")
print(patient_labels["Target"].value_counts().sort_index())

In [ ]:
labels = pd.read_csv(LABELS_CSV)

print("Original CSV shape:", labels.shape)
print("Columns:", labels.columns.tolist())

patient_labels = (
    labels.groupby("patientId", as_index=False)["Target"]
    .max()
)

patient_labels["Target"] = patient_labels["Target"].astype("int32")

patient_labels["image_path"] = patient_labels["patientId"].map(
    lambda patient_id: os.path.join(
        TRAIN_DIR, f"{patient_id}.dcm"
    )
)

patient_labels["image_exists"] = (
    patient_labels["image_path"].map(os.path.isfile)
)

print("\nUnique patients:", len(patient_labels))
print("Images found:", int(patient_labels["image_exists"].sum()))
print("Images missing:", int((~patient_labels["image_exists"]).sum()))

print("\nPatient-level class distribution:")
print(patient_labels["Target"].value_counts().sort_index())

In [ ]:
patient_labels = patient_labels[
    patient_labels["image_exists"]
].copy()

train_df, val_df = train_test_split(
    patient_labels,
    test_size=0.20,
    random_state=SEED,
    stratify=patient_labels["Target"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training patients:", len(train_df))
print("Validation patients:", len(val_df))

print("\nTraining distribution:")
print(train_df["Target"].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df["Target"].value_counts().sort_index())

In [ ]:
IMG_SIZE = 224

def load_dicom_image(image_path):
    dicom = pydicom.dcmread(image_path)
    image = dicom.pixel_array.astype(np.float32)

    slope = float(getattr(dicom, "RescaleSlope", 1))
    intercept = float(getattr(dicom, "RescaleIntercept", 0))

    image = image * slope + intercept

    if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
        image = image.max() + image.min() - image

    low, high = np.percentile(image, [1, 99])

    if high > low:
        image = np.clip(image, low, high)
        image = (image - low) / (high - low)
    else:
        image = np.zeros_like(image, dtype=np.float32)

    image = cv2.resize(
        image,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_AREA
    )

    image = np.repeat(
        image[..., np.newaxis],
        3,
        axis=-1
    )

    return image.astype(np.float32)

In [ ]:
sample_df = train_df.sample(
    n=min(4, len(train_df)),
    random_state=SEED
)

fig, axes = plt.subplots(1, len(sample_df), figsize=(14, 4))

if len(sample_df) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    image = load_dicom_image(row["image_path"])

    ax.imshow(image[:, :, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        "Positive target" if row["Target"] == 1
        else "Negative target"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def load_image_for_tf(image_path, label):
    def decode_image(path):
        path = path.decode("utf-8")
        return load_dicom_image(path)

    image = tf.numpy_function(
        func=decode_image,
        inp=[image_path],
        Tout=tf.float32
    )

    image.set_shape((IMG_SIZE, IMG_SIZE, 3))
    label = tf.cast(label, tf.float32)

    return image, label


def make_dataset(dataframe, training=False):
    image_paths = dataframe["image_path"].astype(str).values
    targets = dataframe["Target"].astype("float32").values

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, targets)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(dataframe), 2048),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image_for_tf,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)

print(
    "Training batches:",
    tf.data.experimental.cardinality(train_ds).numpy()
)

print(
    "Validation batches:",
    tf.data.experimental.cardinality(val_ds).numpy()
)

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["Target"].values
)

class_weight_dict = {
    0: float(class_weights[0]),
    1: float(class_weights[1])
}

print("Class weights:", class_weight_dict)

In [ ]:
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="xray_input")

x = Rescaling(
    scale=2.0,
    offset=-1.0,
    name="input_rescaling"
)(inputs)

base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

x = base_model(x, training=False)
x = GlobalAveragePooling2D(name="global_average_pooling")(x)
x = Dropout(0.35, name="dropout")(x)

outputs = Dense(
    1,
    activation="sigmoid",
    name="pneumonia_output"
)(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        BinaryAccuracy(name="accuracy"),
        AUC(name="auc"),
        Precision(name="precision"),
        Recall(name="recall")
    ]
)

model.summary()

In [ ]:
CHECKPOINT_PATH = (
    "/kaggle/working/pneumonia_densenet121.keras"
)

callbacks = [
    ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="Training Loss")
axes[0].plot(history.history["val_loss"], label="Validation Loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["auc"], label="Training AUC")
axes[1].plot(history.history["val_auc"], label="Validation AUC")
axes[1].set_title("ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

best_model = tf.keras.models.load_model(CHECKPOINT_PATH)

y_true = val_df["Target"].to_numpy()

y_scores = best_model.predict(
    val_ds,
    verbose=1
).ravel()

print("Validation ROC-AUC:", round(roc_auc_score(y_true, y_scores), 4))

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

for threshold in [0.20, 0.30, 0.50]:
    y_pred = (y_scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    print(f"\nThreshold: {threshold:.2f}")
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("Specificity:", round(specificity, 4))
    print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("Confusion matrix:")
    print(np.array([[tn, fp], [fn, tp]]))

In [ ]:
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.metrics import (
    BinaryAccuracy,
    AUC,
    Precision,
    Recall
)

# Unfreeze the backbone, then selectively train the final block
base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = layer.name.startswith("conv5_block")

# Keep BatchNormalization layers frozen
for layer in base_model.layers:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False

# Compile again after changing trainable layers
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        BinaryAccuracy(name="accuracy"),
        AUC(name="auc"),
        Precision(name="precision"),
        Recall(name="recall")
    ]
)

FINE_TUNED_PATH = (
    "/kaggle/working/pneumonia_densenet121_finetuned.keras"
)

fine_tune_callbacks = [
    ModelCheckpoint(
        filepath=FINE_TUNED_PATH,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=fine_tune_callbacks
)

In [ ]:

import tensorflow as tf

strategy = tf.distribute.MirroredStrategy()

print("GPUs detected:", len(tf.config.list_physical_devices("GPU")))
print("GPUs used by strategy:", strategy.num_replicas_in_sync)

In [ ]:
import numpy as np
import tensorflow as tf

from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

fine_tuned_model = tf.keras.models.load_model(FINE_TUNED_PATH)

y_true = val_df["Target"].to_numpy()
y_scores_ft = fine_tuned_model.predict(val_ds, verbose=1).ravel()

print(
    "Fine-tuned validation ROC-AUC:",
    round(roc_auc_score(y_true, y_scores_ft), 4)
)

for threshold in [0.20, 0.30, 0.50]:
    y_pred = (y_scores_ft >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    print(f"\nThreshold: {threshold:.2f}")
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("Specificity:", round(specificity, 4))
    print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("Confusion matrix:")
    print(np.array([[tn, fp], [fn, tp]]))

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# 1. Get true labels from validation dataset
y_true = np.concatenate([
    labels.numpy().reshape(-1)
    for _, labels in val_ds
]).astype(int)

# 2. Predict pneumonia probabilities
y_prob = model.predict(val_ds, verbose=1).reshape(-1)

# 3. Verify predictions and labels are aligned
if len(y_true) != len(y_prob):
    raise ValueError(
        f"Label count ({len(y_true)}) does not match "
        f"prediction count ({len(y_prob)})."
    )

if not np.isin(y_true, [0, 1]).all():
    raise ValueError("Expected binary labels: 0 and 1.")

# 4. Calculate Precision-Recall curve and Average Precision
precision, recall, thresholds = precision_recall_curve(
    y_true, y_prob
)

average_precision = average_precision_score(y_true, y_prob)

# 5. Find threshold with the highest F1-score
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-8)
)

best_index = np.argmax(f1_scores)
best_threshold = float(thresholds[best_index])

# 6. Generate predictions at the selected threshold
y_pred = (y_prob >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

best_precision = precision_score(
    y_true, y_pred, zero_division=0
)
best_recall = recall_score(
    y_true, y_pred, zero_division=0
)
best_f1 = f1_score(
    y_true, y_pred, zero_division=0
)
specificity = tn / (tn + fp) if (tn + fp) else 0.0

# 7. Print results
print("\n========== MODEL EVALUATION ==========")
print(f"Validation samples : {len(y_true)}")
print(f"Average Precision  : {average_precision:.4f}")
print(f"Best F1 threshold  : {best_threshold:.4f}")
print(f"Precision          : {best_precision:.4f}")
print(f"Recall / Sensitivity: {best_recall:.4f}")
print(f"Specificity        : {specificity:.4f}")
print(f"F1-score           : {best_f1:.4f}")

print("\nConfusion Matrix:")
print(np.array([[tn, fp], [fn, tp]]))
print("\nRows = Actual [Negative, Positive]")
print("Columns = Predicted [Negative, Positive]")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["No Pneumonia", "Pneumonia"],
    zero_division=0
))

# 8. Plot Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(
    recall,
    precision,
    label=f"Average Precision = {average_precision:.4f}"
)
plt.scatter(
    best_recall,
    best_precision,
    label=f"Best F1 threshold = {best_threshold:.3f}"
)
plt.xlabel("Recall (Sensitivity)")
plt.ylabel("Precision")
plt.title("DenseNet121 Precision-Recall Curve")
plt.legend()
plt.grid(True)
plt.show()

# 9. Plot confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix=np.array([[tn, fp], [fn, tp]]),
    display_labels=["No Pneumonia", "Pneumonia"]
).plot(cmap="Blues", values_format="d")

plt.title(f"Confusion Matrix (Threshold = {best_threshold:.3f})")
plt.show()


In [1]:

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from PIL import Image
from io import BytesIO
from IPython.display import display
import ipywidgets as widgets

# Upload interface
uploader = widgets.FileUpload(
    accept=".jpg,.jpeg,.png",
    multiple=False,
    description="Upload X-ray"
)

button = widgets.Button(
    description="Generate Grad-CAM",
    button_style="primary"
)

output = widgets.Output()


def make_gradcam(image_batch, model):
    # Find the DenseNet121 base model
    base_model = next(
        (layer for layer in model.layers
         if isinstance(layer, tf.keras.Model)),
        None
    )

    if base_model is None:
        raise ValueError("DenseNet121 base model not found.")

    # Last convolutional layer
    conv_layer = base_model.get_layer("conv5_block16_2_conv")

    grad_model = tf.keras.Model(
        inputs=base_model.inputs,
        outputs=[conv_layer.output, base_model.output]
    )

    # Calculate gradients of the pneumonia score
    with tf.GradientTape() as tape:
        conv_outputs, base_outputs = grad_model(
            image_batch, training=False
        )

        pooled = model.layers[-3](base_outputs)
        dropped = model.layers[-2](pooled)
        predictions = model.layers[-1](dropped)

        score = predictions[:, 0]

    gradients = tape.gradient(score, conv_outputs)

    if gradients is None:
        raise ValueError(
            "Gradients unavailable. Check the model architecture."
        )

    weights = tf.reduce_mean(gradients, axis=(1, 2))

    cam = tf.reduce_sum(
        weights[:, tf.newaxis, tf.newaxis, :] * conv_outputs,
        axis=-1
    )[0]

    cam = tf.maximum(cam, 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)

    return cam.numpy()


def show_gradcam(_):
    with output:
        output.clear_output()

        if not uploader.value:
            print("Please upload a chest X-ray first.")
            return

        try:
            # Read uploaded file
            uploaded = uploader.value

            if isinstance(uploaded, dict):
                file_info = next(iter(uploaded.values()))
                image_bytes = file_info["content"]
            else:
                file_info = uploaded[0]
                image_bytes = file_info["content"]

            # Load image
            original = Image.open(
                BytesIO(image_bytes)
            ).convert("RGB")

            resized = original.resize((224, 224))

            image_array = np.asarray(
                resized, dtype=np.float32
            ) / 255.0

            image_batch = np.expand_dims(image_array, axis=0)

            # Model prediction
            probability = float(
                model.predict(image_batch, verbose=0)[0][0]
            )

            # Generate Grad-CAM
            cam = make_gradcam(image_batch, model)

            cam_image = Image.fromarray(
                np.uint8(cam * 255)
            ).resize((224, 224))

            cam_array = np.asarray(
                cam_image, dtype=np.float32
            ) / 255.0

            # Display three visualizations
            fig, axes = plt.subplots(1, 3, figsize=(16, 5))

            axes[0].imshow(resized)
            axes[0].set_title("1. Original X-ray")

            axes[1].imshow(
                cam_array, cmap="jet", vmin=0, vmax=1
            )
            axes[1].set_title("2. Grad-CAM Heatmap")

            axes[2].imshow(resized)
            axes[2].imshow(
                cam_array,
                cmap="jet",
                alpha=0.45,
                vmin=0,
                vmax=1
            )
            axes[2].set_title("3. Grad-CAM Overlay")

            for ax in axes:
                ax.axis("off")

            plt.tight_layout()
            plt.show()

            # Display prediction
            print(f"Pneumonia probability: {probability:.2%}")

            if probability >= 0.307:
                print("Model output: Pneumonia detected")
            else:
                print("Model output: No pneumonia detected")

            print(
                "\nExperimental prediction only. ")

        except Exception as error:
            print(f"Error: {error}")


button.on_click(show_gradcam)

display(uploader, button, output)


FileUpload(value=(), accept='.jpg,.jpeg,.png', description='Upload X-ray')

Button(button_style='primary', description='Generate Grad-CAM', style=ButtonStyle())

Output()